# IA — Aula 5 — Aprendizado de Máquina Clássico
## Trabalho da semana: Filtro de Spam — Spambase (UCI)

**T5-A1 — Ambiente, carga e inspeção da base**

Objetivos desta etapa:

1. Preparar o ambiente de execução.
2. Carregar a base **Spambase (UCI)**.
3. Confirmar as dimensões esperadas: **4.601 registros e 57 atributos preditores**.
4. Inspecionar a variável-alvo.
5. Verificar distribuição das classes, valores ausentes e tipos.
6. Confirmar a existência dos atributos `george`, `650`, `hp` e `hpl`, que serão usados em uma etapa posterior do trabalho.

> Nesta etapa **não treinaremos modelos ainda**. O objetivo é garantir que a base e o pipeline experimental estejam corretos antes dos experimentos.


## 1. Configuração do ambiente

A célula abaixo instala as dependências necessárias no ambiente atual.

> Em Google Colab ou Jupyter, execute esta célula uma vez. Se os pacotes já estiverem instalados, a operação será rápida.


In [1]:
%pip install -q ucimlrepo pandas numpy scikit-learn matplotlib


## 2. Imports e configurações de reprodutibilidade

Usaremos `random_state = 42` nas próximas etapas para tornar os experimentos reproduzíveis.


In [2]:
from __future__ import annotations

import sys
import platform
import numpy as np
import pandas as pd

from ucimlrepo import fetch_ucirepo

RANDOM_STATE = 42

print("Python:", sys.version.split()[0])
print("Plataforma:", platform.platform())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("random_state:", RANDOM_STATE)


Python: 3.13.15
Plataforma: Linux-6.6.122+-x86_64-with-glibc2.39
NumPy: 2.1.3
Pandas: 2.2.3
random_state: 42


## 3. Carregamento da Spambase

A base será obtida pelo pacote `ucimlrepo`, usando o identificador **94** da UCI Machine Learning Repository.


In [3]:
spambase = fetch_ucirepo(id=94)

X = spambase.data.features.copy()
y = spambase.data.targets.squeeze().copy()

# Garantir rótulo inteiro (0 = legítimo, 1 = spam)
y = pd.to_numeric(y, errors="raise").astype(int)
y.name = "spam"

print("Dataset carregado com sucesso.")
print("X:", X.shape)
print("y:", y.shape)


Dataset carregado com sucesso.
X: (4601, 57)
y: (4601,)


## 4. Validação estrutural da base

Segundo o enunciado do trabalho, esperamos:

- **4.601 e-mails**
- **57 atributos numéricos**
- alvo binário: **0 = legítimo**, **1 = spam**

As asserções abaixo fazem o notebook interromper a execução caso a estrutura recebida seja diferente da esperada.


In [4]:
EXPECTED_ROWS = 4601
EXPECTED_FEATURES = 57

assert X.shape[0] == EXPECTED_ROWS, (
    f"Número inesperado de registros: {X.shape[0]} (esperado: {EXPECTED_ROWS})"
)
assert X.shape[1] == EXPECTED_FEATURES, (
    f"Número inesperado de atributos: {X.shape[1]} (esperado: {EXPECTED_FEATURES})"
)
assert len(y) == EXPECTED_ROWS, (
    f"Tamanho inesperado do alvo: {len(y)} (esperado: {EXPECTED_ROWS})"
)
assert set(y.unique()).issubset({0, 1}), (
    f"Alvo não binário como esperado. Valores encontrados: {sorted(y.unique())}"
)

print("✓ 4.601 registros confirmados")
print("✓ 57 atributos preditores confirmados")
print("✓ alvo binário confirmado")


✓ 4.601 registros confirmados
✓ 57 atributos preditores confirmados
✓ alvo binário confirmado


## 5. Primeira inspeção dos dados


In [5]:
display(X.head())

print("\nTipos dos atributos:")
display(X.dtypes.value_counts().rename("quantidade").to_frame())

print("\nPrimeiros nomes de atributos:")
print(X.columns[:15].tolist())


,word_freq_make,word_freq_address,word_freq_all,word_freq_3d,word_freq_our,word_freq_over,word_freq_remove,word_freq_internet,word_freq_order,word_freq_mail,...,word_freq_conference,char_freq_;,char_freq_(,char_freq_[,char_freq_!,char_freq_$,char_freq_#,capital_run_length_average,capital_run_length_longest,capital_run_length_total
0,0.00,0.64,0.64,0.0,0.32,0.00,0.00,0.00,0.00,0.00,...,0.0,0.00,0.000,0.0,0.778,0.000,0.000,3.756,61,278
1,0.21,0.28,0.50,0.0,0.14,0.28,0.21,0.07,0.00,0.94,...,0.0,0.00,0.132,0.0,0.372,0.180,0.048,5.114,101,1028
2,0.06,0.00,0.71,0.0,1.23,0.19,0.19,0.12,0.64,0.25,...,0.0,0.01,0.143,0.0,0.276,0.184,0.010,9.821,485,2259
3,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.0,0.00,0.137,0.0,0.137,0.000,0.000,3.537,40,191
4,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.0,0.00,0.135,0.0,0.135,0.000,0.000,3.537,40,191



Tipos dos atributos:


,quantidade
float64,55
int64,2



Primeiros nomes de atributos:
['word_freq_make', 'word_freq_address', 'word_freq_all', 'word_freq_3d', 'word_freq_our', 'word_freq_over', 'word_freq_remove', 'word_freq_internet', 'word_freq_order', 'word_freq_mail', 'word_freq_receive', 'word_freq_will', 'word_freq_people', 'word_freq_report', 'word_freq_addresses']


## 6. Distribuição da variável-alvo

Uma das primeiras verificações em classificação é entender o balanço das classes.

Nesta base, esperamos aproximadamente **39% spam** e **61% e-mails legítimos**.

Essa distribuição será importante posteriormente para:

- construir o baseline;
- usar divisão estratificada;
- interpretar a acurácia corretamente;
- avaliar o custo dos falsos positivos.


In [6]:
class_counts = y.value_counts().sort_index()
class_pct = y.value_counts(normalize=True).sort_index().mul(100)

class_summary = pd.DataFrame({
    "classe": ["legítimo", "spam"],
    "quantidade": [class_counts.get(0, 0), class_counts.get(1, 0)],
    "percentual": [class_pct.get(0, 0), class_pct.get(1, 0)],
}, index=[0, 1])

class_summary.index.name = "rotulo"
display(class_summary)

spam_rate = class_pct.get(1, np.nan)
legit_rate = class_pct.get(0, np.nan)

print(f"Legítimos: {legit_rate:.2f}%")
print(f"Spam:      {spam_rate:.2f}%")


,classe,quantidade,percentual
rotulo,,,
0,legítimo,2788,60.595523
1,spam,1813,39.404477


Legítimos: 60.60%
Spam:      39.40%


## 7. Qualidade básica dos dados

Antes de qualquer modelagem, verificamos:

- valores ausentes;
- infinitos;
- duplicatas;
- atributos constantes.

Essas verificações são simples, mas ajudam a evitar conclusões erradas nas etapas seguintes.


In [7]:
missing_total = int(X.isna().sum().sum())
inf_total = int(np.isinf(X.to_numpy(dtype=float)).sum())
duplicate_rows = int(pd.concat([X, y], axis=1).duplicated().sum())
constant_features = X.columns[X.nunique(dropna=False) <= 1].tolist()

quality_summary = pd.DataFrame({
    "checagem": [
        "valores ausentes",
        "valores infinitos",
        "linhas duplicadas (X + y)",
        "atributos constantes",
    ],
    "quantidade": [
        missing_total,
        inf_total,
        duplicate_rows,
        len(constant_features),
    ],
})

display(quality_summary)

if constant_features:
    print("Atributos constantes:", constant_features)


,checagem,quantidade
0,valores ausentes,0
1,valores infinitos,0
2,linhas duplicadas (X + y),391
3,atributos constantes,0


## 8. Estatísticas descritivas

As 57 variáveis são numéricas, mas não necessariamente estão na mesma escala.

Isso será especialmente importante no experimento com **k-NN**, porque esse algoritmo depende de distância entre observações.


In [8]:
summary = X.describe().T
summary["range"] = summary["max"] - summary["min"]

display(summary.head(15))

print("\nAtributos com maior amplitude:")
display(
    summary[["min", "max", "mean", "std", "range"]]
    .sort_values("range", ascending=False)
    .head(10)
)


,count,mean,std,min,25%,50%,75%,max,range
word_freq_make,4601.0,0.104553,0.305358,0.0,0.0,0.0,0.00,4.54,4.54
word_freq_address,4601.0,0.213015,1.290575,0.0,0.0,0.0,0.00,14.28,14.28
word_freq_all,4601.0,0.280656,0.504143,0.0,0.0,0.0,0.42,5.10,5.10
word_freq_3d,4601.0,0.065425,1.395151,0.0,0.0,0.0,0.00,42.81,42.81
word_freq_our,4601.0,0.312223,0.672513,0.0,0.0,0.0,0.38,10.00,10.00
word_freq_over,4601.0,0.095901,0.273824,0.0,0.0,0.0,0.00,5.88,5.88
word_freq_remove,4601.0,0.114208,0.391441,0.0,0.0,0.0,0.00,7.27,7.27
word_freq_internet,4601.0,0.105295,0.401071,0.0,0.0,0.0,0.00,11.11,11.11
word_freq_order,4601.0,0.090067,0.278616,0.0,0.0,0.0,0.00,5.26,5.26
word_freq_mail,4601.0,0.239413,0.644755,0.0,0.0,0.0,0.16,18.18,18.18



Atributos com maior amplitude:


,min,max,mean,std,range
capital_run_length_total,1.0,15841.000,283.289285,606.347851,15840.000
capital_run_length_longest,1.0,9989.000,52.172789,194.891310,9988.000
capital_run_length_average,1.0,1102.500,5.191515,31.729449,1101.500
word_freq_3d,0.0,42.810,0.065425,1.395151,42.810
word_freq_george,0.0,33.330,0.767305,3.367292,33.330
char_freq_!,0.0,32.478,0.269071,0.815672,32.478
word_freq_edu,0.0,22.050,0.179824,0.911119,22.050
word_freq_re,0.0,21.420,0.301224,1.011687,21.420
word_freq_hp,0.0,20.830,0.549504,1.671349,20.830
word_freq_85,0.0,20.000,0.105412,0.532260,20.000


## 9. Verificação dos quatro atributos do experimento de generalização

O item 5 do trabalho exige retirar os atributos associados a:

- `george`
- `650`
- `hp`
- `hpl`

Na UCI, os nomes completos esperados são:

- `word_freq_george`
- `word_freq_650`
- `word_freq_hp`
- `word_freq_hpl`


In [9]:
SPECIAL_FEATURES = [
    "word_freq_george",
    "word_freq_650",
    "word_freq_hp",
    "word_freq_hpl",
]

special_feature_check = pd.DataFrame({
    "atributo": SPECIAL_FEATURES,
    "presente": [feature in X.columns for feature in SPECIAL_FEATURES],
})

display(special_feature_check)

missing_special = [
    feature for feature in SPECIAL_FEATURES
    if feature not in X.columns
]

assert not missing_special, (
    "Atributos esperados não encontrados: "
    + ", ".join(missing_special)
)

print("✓ Os quatro atributos necessários para o experimento 5 estão presentes.")


,atributo,presente
0,word_freq_george,True
1,word_freq_650,True
2,word_freq_hp,True
3,word_freq_hpl,True


✓ Os quatro atributos necessários para o experimento 5 estão presentes.


## 10. Metadados do dataset

Esta célula registra informações fornecidas pelo repositório UCI. É útil para documentar a proveniência da base no notebook.


In [10]:
print("METADADOS")
print("-" * 80)
print(spambase.metadata)

print("\nVARIÁVEIS")
print("-" * 80)
display(spambase.variables)


METADADOS
--------------------------------------------------------------------------------
{'uci_id': 94, 'name': 'Spambase', 'repository_url': 'https://archive.ics.uci.edu/dataset/94/spambase', 'data_url': 'https://archive.ics.uci.edu/static/public/94/data.csv', 'abstract': 'Classifying Email as Spam or Non-Spam', 'area': 'Computer Science', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 4601, 'num_features': 57, 'feature_types': ['Integer', 'Real'], 'demographics': [], 'target_col': ['Class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1999, 'last_updated': 'Mon Aug 28 2023', 'dataset_doi': '10.24432/C53G6X', 'creators': ['Mark Hopkins', 'Erik Reeber', 'George Forman', 'Jaap Suermondt'], 'intro_paper': None, 'additional_info': {'summary': 'The "spam" concept is diverse: advertisements for products/web sites, make money fast schemes, chain letters, pornography...\n\nThe classification ta

,name,role,type,demographic,description,units,missing_values
0,word_freq_make,Feature,Continuous,None,None,None,no
1,word_freq_address,Feature,Continuous,None,None,None,no
2,word_freq_all,Feature,Continuous,None,None,None,no
3,word_freq_3d,Feature,Continuous,None,None,None,no
4,word_freq_our,Feature,Continuous,None,None,None,no
5,word_freq_over,Feature,Continuous,None,None,None,no
6,word_freq_remove,Feature,Continuous,None,None,None,no
7,word_freq_internet,Feature,Continuous,None,None,None,no
8,word_freq_order,Feature,Continuous,None,None,None,no
9,word_freq_mail,Feature,Continuous,None,None,None,no


## 11. Checklist da T5-A1

Ao finalizar esta etapa, devemos ter confirmado:

- [x] ambiente configurado;
- [x] Spambase carregada;
- [x] 4.601 registros;
- [x] 57 atributos preditores;
- [x] alvo binário;
- [x] distribuição das classes conhecida;
- [x] qualidade básica dos dados inspecionada;
- [x] escalas dos atributos inspecionadas;
- [x] `george`, `650`, `hp` e `hpl` localizados.

### Próxima etapa — T5-A2

Construir o **baseline que classifica todo e-mail como legítimo** e calcular sua acurácia e demais métricas relevantes.


---
## T5-A2 — Baseline: sempre prever e-mail legítimo

O primeiro item do trabalho pede um **baseline que sempre prevê `"legítimo"`**.

O baseline estabelece uma referência mínima para os modelos que serão avaliados posteriormente. Como a classe majoritária da Spambase é `0 = legítimo`, um classificador que ignore completamente os atributos e sempre produza `0` terá uma acurácia igual à proporção de e-mails legítimos da base.

Entretanto, **acurácia isolada pode ser enganosa**: esse baseline não identifica nenhum spam. Por isso, além da acurácia, calcularemos a matriz de confusão, precision, recall e F1 para a classe positiva (`spam = 1`).


### 12. Imports para avaliação

Nesta etapa não há treinamento. Apenas construiremos uma predição constante e mediremos seu comportamento.


In [11]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)


### 13. Construção do baseline

O vetor `y_baseline` terá o mesmo tamanho de `y`, mas todas as previsões serão `0`, isto é, **legítimo**.


In [12]:
# Baseline: classificar todos os e-mails como legítimos (classe 0)
y_baseline = np.zeros(len(y), dtype=int)

assert len(y_baseline) == len(y)
assert set(np.unique(y_baseline)) == {0}

print(f"Total de previsões: {len(y_baseline)}")
print("Classe prevista pelo baseline: 0 = legítimo")


Total de previsões: 4601
Classe prevista pelo baseline: 0 = legítimo


### 14. Métricas do baseline

A classe positiva do problema é `1 = spam`.

Para evitar avisos ao calcular precision quando o modelo não prevê nenhum spam, utilizamos `zero_division=0`.


In [13]:
baseline_accuracy = accuracy_score(y, y_baseline)
baseline_precision = precision_score(y, y_baseline, pos_label=1, zero_division=0)
baseline_recall = recall_score(y, y_baseline, pos_label=1, zero_division=0)
baseline_f1 = f1_score(y, y_baseline, pos_label=1, zero_division=0)

baseline_results = pd.DataFrame({
    "modelo": ["Baseline — sempre legítimo"],
    "accuracy": [baseline_accuracy],
    "precision_spam": [baseline_precision],
    "recall_spam": [baseline_recall],
    "f1_spam": [baseline_f1],
})

display(
    baseline_results.style.format({
        "accuracy": "{:.4f}",
        "precision_spam": "{:.4f}",
        "recall_spam": "{:.4f}",
        "f1_spam": "{:.4f}",
    })
)

print(f"Acurácia do baseline: {baseline_accuracy:.2%}")
print(f"Recall de spam:       {baseline_recall:.2%}")


,modelo,accuracy,precision_spam,recall_spam,f1_spam
0,Baseline — sempre legítimo,0.6060,0.0000,0.0000,0.0000


Acurácia do baseline: 60.60%
Recall de spam:       0.00%


### 15. Matriz de confusão

Usaremos a convenção do `scikit-learn`:

\[
\begin{bmatrix}
TN & FP \\
FN & TP
\end{bmatrix}
\]

onde:

- **TN**: legítimo corretamente classificado como legítimo;
- **FP**: legítimo incorretamente classificado como spam;
- **FN**: spam incorretamente classificado como legítimo;
- **TP**: spam corretamente classificado como spam.


In [14]:
cm = confusion_matrix(y, y_baseline, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

confusion_df = pd.DataFrame(
    cm,
    index=["Real: legítimo (0)", "Real: spam (1)"],
    columns=["Previsto: legítimo (0)", "Previsto: spam (1)"],
)

display(confusion_df)

print(f"TN (legítimos corretamente aceitos): {tn}")
print(f"FP (legítimos marcados como spam):   {fp}")
print(f"FN (spams que passam como legítimos): {fn}")
print(f"TP (spams detectados):                {tp}")


,Previsto: legítimo (0),Previsto: spam (1)
Real: legítimo (0),2788,0
Real: spam (1),1813,0


TN (legítimos corretamente aceitos): 2788
FP (legítimos marcados como spam):   0
FN (spams que passam como legítimos): 1813
TP (spams detectados):                0


### 16. Relatório de classificação

Este relatório explicita por que a acurácia, sozinha, não é suficiente para avaliar o filtro.


In [15]:
print(
    classification_report(
        y,
        y_baseline,
        labels=[0, 1],
        target_names=["legítimo", "spam"],
        zero_division=0,
    )
)


              precision    recall  f1-score   support

    legítimo       0.61      1.00      0.75      2788
        spam       0.00      0.00      0.00      1813

    accuracy                           0.61      4601
   macro avg       0.30      0.50      0.38      4601
weighted avg       0.37      0.61      0.46      4601



### 17. Interpretação do resultado

O baseline obtém uma acurácia equivalente à proporção da classe majoritária (`legítimo`). Apesar disso, seu desempenho como **filtro de spam é nulo**, pois:

- não prevê nenhum e-mail como spam;
- `recall` da classe spam = 0;
- todos os spams são falsos negativos;
- nenhum spam é efetivamente bloqueado.

Essa etapa mostra por que **acurácia não deve ser analisada isoladamente em problemas de classificação**, especialmente quando as classes são desbalanceadas ou quando os diferentes tipos de erro possuem custos distintos.

O baseline será usado como referência mínima para os modelos avaliados nas próximas etapas.


### Checkpoint T5-A2

Ao executar esta etapa no **Google Colab**, registre os valores obtidos. Eles serão reutilizados no relatório final de uma página.

**Próxima etapa — T5-A3:** criar uma divisão **train/test estratificada**, preservar um conjunto de teste final e configurar a **validação cruzada estratificada de 5 dobras** para os modelos do trabalho.


---
## T5-A3 — Split estratificado + StratifiedKFold de 5 dobras

Nesta etapa será preparada a infraestrutura experimental usada na comparação dos modelos.

A estratégia adotada será:

1. separar a base original em **treino (80%)** e **teste final (20%)**, preservando a proporção entre spam e e-mails legítimos por meio de `stratify=y`;
2. manter o conjunto de teste **isolado**, sem utilizá-lo para seleção de modelos ou ajuste de hiperparâmetros;
3. executar a comparação dos modelos somente no conjunto de treino por meio de **validação cruzada estratificada de 5 dobras** (`StratifiedKFold`).

Essa organização evita que o conjunto de teste final influencie as decisões tomadas durante o desenvolvimento do modelo.


### 18. Imports para divisão e validação cruzada

In [16]:
from sklearn.model_selection import train_test_split, StratifiedKFold

### 19. Divisão treino/teste estratificada

Usaremos:

- `test_size=0.20`
- `random_state=42`
- `stratify=y`

A estratificação garante que as proporções das classes permaneçam aproximadamente iguais nos dois subconjuntos.


In [17]:
TEST_SIZE = 0.20

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Dimensões:")
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")


Dimensões:
X_train: (3680, 57)
X_test:  (921, 57)
y_train: (3680,)
y_test:  (921,)


### 20. Verificação da estratificação

Vamos comparar a proporção de spam na base completa, no conjunto de treino e no conjunto de teste.


In [18]:
def class_distribution(target: pd.Series) -> pd.Series:
    return target.value_counts(normalize=True).sort_index().mul(100)

distribution_summary = pd.DataFrame({
    "base_completa_%": class_distribution(y),
    "treino_%": class_distribution(y_train),
    "teste_%": class_distribution(y_test),
}).rename(index={0: "legítimo (0)", 1: "spam (1)"})

display(distribution_summary.style.format("{:.2f}%"))


,base_completa_%,treino_%,teste_%
spam,,,
legítimo (0),60.60%,60.60%,60.59%
spam (1),39.40%,39.40%,39.41%


### 21. Verificações de integridade do split

As asserções abaixo garantem que:

- treino e teste não compartilham índices;
- a soma das amostras de treino e teste corresponde à base original;
- ambos os subconjuntos contêm as duas classes.


In [19]:
assert set(X_train.index).isdisjoint(set(X_test.index)), (
    "Há sobreposição de índices entre treino e teste."
)

assert len(X_train) + len(X_test) == len(X), (
    "A soma de treino e teste não corresponde ao tamanho da base original."
)

assert set(y_train.unique()) == {0, 1}, "Treino não contém as duas classes."
assert set(y_test.unique()) == {0, 1}, "Teste não contém as duas classes."

print("✓ treino e teste sem sobreposição de índices")
print("✓ número total de registros preservado")
print("✓ ambas as classes presentes em treino e teste")


✓ treino e teste sem sobreposição de índices
✓ número total de registros preservado
✓ ambas as classes presentes em treino e teste


### 22. Configuração da validação cruzada estratificada

A atividade exige **validação cruzada de 5 dobras**.

Usaremos:

```python
StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
```

`StratifiedKFold` preserva aproximadamente a proporção das classes em cada dobra de treino e validação.


In [20]:
N_SPLITS = 5

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

print(cv)


StratifiedKFold(n_splits=5, random_state=42, shuffle=True)


### 23. Inspeção das 5 dobras

Antes de treinar os modelos, verificaremos o tamanho e a distribuição de spam em cada fold.

O conjunto de **teste final não participa dessas dobras**.


In [21]:
fold_rows = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X_train, y_train), start=1):
    y_fold_train = y_train.iloc[train_idx]
    y_fold_valid = y_train.iloc[valid_idx]

    fold_rows.append({
        "fold": fold,
        "n_treino": len(train_idx),
        "n_validacao": len(valid_idx),
        "spam_treino_%": y_fold_train.mean() * 100,
        "spam_validacao_%": y_fold_valid.mean() * 100,
    })

fold_summary = pd.DataFrame(fold_rows)

display(
    fold_summary.style.format({
        "spam_treino_%": "{:.2f}%",
        "spam_validacao_%": "{:.2f}%",
    })
)


,fold,n_treino,n_validacao,spam_treino_%,spam_validacao_%
0,1,2944,736,39.40%,39.40%
1,2,2944,736,39.40%,39.40%
2,3,2944,736,39.40%,39.40%
3,4,2944,736,39.40%,39.40%
4,5,2944,736,39.40%,39.40%


### 24. Interpretação metodológica

A partir deste ponto:

- `X_train` e `y_train` serão usados para **comparar e selecionar modelos**;
- `cv` definirá as **5 dobras estratificadas**;
- `X_test` e `y_test` permanecerão reservados para a **avaliação final**;
- nenhum resultado do conjunto de teste será usado para escolher modelo, hiperparâmetros ou limiar.

Essa separação ajuda a evitar uma estimativa excessivamente otimista do desempenho de generalização.

> Observação: a análise inicial identificou linhas duplicadas na base original. Elas serão mantidas no experimento principal para preservar os 4.601 registros do enunciado. Essa característica será registrada como limitação metodológica e poderá ser explorada posteriormente em análise de sensibilidade.


### Checkpoint T5-A3

Ao finalizar esta etapa no Google Colab, confirme:

- [ ] treino e teste foram criados;
- [ ] as proporções de spam e legítimo permanecem semelhantes nos dois conjuntos;
- [ ] não há sobreposição de índices;
- [ ] as 5 dobras apresentam proporções de classe semelhantes;
- [ ] o conjunto de teste final continua intocado.

**Próxima etapa — T5-A4:** comparar pelo menos três modelos vistos em aula usando as mesmas 5 dobras e reportar **média e desvio-padrão** das métricas.


---
## T5-A4 — Comparação de modelos com validação cruzada

O item 2 do trabalho solicita **pelo menos três modelos**, utilizando divisão estratificada e **validação cruzada de 5 dobras**, com reporte de **média e desvio-padrão**.

Nesta etapa compararemos quatro modelos clássicos:

1. **Regressão Logística**
2. **Árvore de Decisão**
3. **k-NN**
4. **Random Forest**

Para garantir uma comparação metodologicamente correta:

- todos os modelos usarão as **mesmas 5 dobras** definidas em `cv`;
- a avaliação será feita **somente em `X_train` e `y_train`**;
- `X_test` e `y_test` continuarão isolados para a avaliação final;
- algoritmos sensíveis à escala, como Regressão Logística e k-NN, usarão `StandardScaler` dentro de um `Pipeline`, evitando vazamento de informação entre as dobras.

> O experimento específico pedido no item 3 — **k-NN com e sem padronização** — será aprofundado na T5-A5. Aqui utilizaremos a versão padronizada do k-NN como representante dessa família.


### 25. Imports para modelagem e avaliação cruzada


In [22]:
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier


### 26. Definição dos modelos

Os hiperparâmetros serão mantidos próximos aos valores padrão, pois o objetivo desta etapa é **comparar famílias de modelos**, e não realizar tuning extensivo.

A padronização fica **dentro do pipeline**, portanto é ajustada novamente em cada fold somente com os dados de treino daquela dobra.


In [23]:
models = {
    "Regressão Logística": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=2000,
            random_state=RANDOM_STATE,
        )),
    ]),

    "Árvore de Decisão": DecisionTreeClassifier(
        random_state=RANDOM_STATE,
    ),

    "k-NN padronizado": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(
            n_neighbors=5,
        )),
    ]),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

for name, model in models.items():
    print(f"{name}:")
    print(model)
    print("-" * 80)


Regressão Logística:
Pipeline(steps=[('scaler', StandardScaler()),
                ('model', LogisticRegression(max_iter=2000, random_state=42))])
--------------------------------------------------------------------------------
Árvore de Decisão:
DecisionTreeClassifier(random_state=42)
--------------------------------------------------------------------------------
k-NN padronizado:
Pipeline(steps=[('scaler', StandardScaler()),
                ('model', KNeighborsClassifier())])
--------------------------------------------------------------------------------
Random Forest:
RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)
--------------------------------------------------------------------------------


### 27. Métricas de avaliação

Além da **acurácia**, que é explicitamente pedida no trabalho, registraremos:

- `precision`: entre os e-mails classificados como spam, quantos realmente eram spam;
- `recall`: entre os spams reais, quantos foram detectados;
- `F1`: média harmônica entre precision e recall;
- `ROC-AUC`: capacidade de separar as duas classes considerando diferentes limiares.

Essas métricas serão úteis porque, neste problema, falsos positivos e falsos negativos possuem consequências diferentes.


In [24]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

print("Métricas:", ", ".join(scoring.keys()))


Métricas: accuracy, precision, recall, f1, roc_auc


### 28. Validação cruzada dos quatro modelos

Cada modelo será avaliado nas **mesmas 5 dobras estratificadas**.

Para cada métrica serão preservados os resultados individuais dos folds e, em seguida, calcularemos:

\[
\text{média} = \frac{1}{5}\sum_{i=1}^{5} x_i
\]

e o respectivo **desvio-padrão** entre as cinco avaliações.


In [25]:
cv_raw_results = {}
fold_records = []

for model_name, model in models.items():
    print(f"Avaliando: {model_name}...")

    scores = cross_validate(
        estimator=model,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
    )

    cv_raw_results[model_name] = scores

    for fold in range(N_SPLITS):
        fold_records.append({
            "modelo": model_name,
            "fold": fold + 1,
            "accuracy": scores["test_accuracy"][fold],
            "precision": scores["test_precision"][fold],
            "recall": scores["test_recall"][fold],
            "f1": scores["test_f1"][fold],
            "roc_auc": scores["test_roc_auc"][fold],
            "fit_time_s": scores["fit_time"][fold],
        })

print("✓ Validação cruzada concluída para todos os modelos.")


Avaliando: Regressão Logística...
Avaliando: Árvore de Decisão...
Avaliando: k-NN padronizado...
Avaliando: Random Forest...
✓ Validação cruzada concluída para todos os modelos.


### 29. Resultados por fold

A tabela abaixo permite verificar não apenas a média final, mas também a **estabilidade** de cada modelo entre as cinco dobras.


In [26]:
cv_folds_df = pd.DataFrame(fold_records)

display(
    cv_folds_df.style.format({
        "accuracy": "{:.4f}",
        "precision": "{:.4f}",
        "recall": "{:.4f}",
        "f1": "{:.4f}",
        "roc_auc": "{:.4f}",
        "fit_time_s": "{:.3f}",
    })
)


,modelo,fold,accuracy,precision,recall,f1,roc_auc,fit_time_s
0,Regressão Logística,1,0.9375,0.9326,0.9069,0.9196,0.9775,0.230
1,Regressão Logística,2,0.9239,0.9120,0.8931,0.9024,0.9656,0.113
2,Regressão Logística,3,0.9307,0.9544,0.8655,0.9078,0.9782,0.057
3,Regressão Logística,4,0.9076,0.8964,0.8655,0.8807,0.9678,0.091
4,Regressão Logística,5,0.9144,0.9068,0.8724,0.8893,0.9634,0.073
5,Árvore de Decisão,1,0.9226,0.9059,0.8966,0.9012,0.9180,0.163
6,Árvore de Decisão,2,0.9280,0.9044,0.9138,0.9091,0.9255,0.333
7,Árvore de Decisão,3,0.9049,0.8548,0.9138,0.8833,0.9064,0.196
8,Árvore de Decisão,4,0.9076,0.8801,0.8862,0.8832,0.9039,0.171
9,Árvore de Decisão,5,0.9049,0.8873,0.8690,0.8780,0.8986,0.268


### 30. Média e desvio-padrão das métricas

Esta é a tabela principal para atender ao item 2 do trabalho.

O desvio-padrão ajuda a avaliar a estabilidade: valores menores indicam que o desempenho variou menos entre os folds.


In [27]:
metrics = ["accuracy", "precision", "recall", "f1", "roc_auc"]

summary_rows = []

for model_name in models.keys():
    model_folds = cv_folds_df[cv_folds_df["modelo"] == model_name]

    row = {"modelo": model_name}

    for metric in metrics:
        row[f"{metric}_media"] = model_folds[metric].mean()
        row[f"{metric}_dp"] = model_folds[metric].std(ddof=1)

    row["fit_time_media_s"] = model_folds["fit_time_s"].mean()
    summary_rows.append(row)

cv_summary = pd.DataFrame(summary_rows).sort_values(
    by="accuracy_media",
    ascending=False,
).reset_index(drop=True)

display(
    cv_summary.style.format({
        col: "{:.4f}"
        for col in cv_summary.columns
        if col != "modelo"
    })
)


,modelo,accuracy_media,accuracy_dp,precision_media,precision_dp,recall_media,recall_dp,f1_media,f1_dp,roc_auc_media,roc_auc_dp,fit_time_media_s
0,Random Forest,0.9527,0.0061,0.9525,0.0050,0.9262,0.0163,0.9391,0.0083,0.9861,0.0041,6.0576
1,Regressão Logística,0.9228,0.0121,0.9204,0.0231,0.8807,0.0185,0.9000,0.0153,0.9705,0.0069,0.1128
2,Árvore de Decisão,0.9136,0.0109,0.8865,0.0209,0.8959,0.0191,0.8910,0.0134,0.9105,0.0110,0.2260
3,k-NN padronizado,0.9068,0.0040,0.9006,0.0164,0.8586,0.0101,0.8790,0.0038,0.9530,0.0069,0.0492


### 31. Tabela compacta — média ± desvio-padrão

Para facilitar a leitura e posterior uso no relatório, apresentaremos as métricas no formato:

**média ± desvio-padrão**


In [28]:
compact_rows = []

for _, row in cv_summary.iterrows():
    compact_rows.append({
        "modelo": row["modelo"],
        "accuracy": f'{row["accuracy_media"]:.4f} ± {row["accuracy_dp"]:.4f}',
        "precision": f'{row["precision_media"]:.4f} ± {row["precision_dp"]:.4f}',
        "recall": f'{row["recall_media"]:.4f} ± {row["recall_dp"]:.4f}',
        "f1": f'{row["f1_media"]:.4f} ± {row["f1_dp"]:.4f}',
        "roc_auc": f'{row["roc_auc_media"]:.4f} ± {row["roc_auc_dp"]:.4f}',
    })

cv_compact = pd.DataFrame(compact_rows)

display(cv_compact)


,modelo,accuracy,precision,recall,f1,roc_auc
0,Random Forest,0.9527 ± 0.0061,0.9525 ± 0.0050,0.9262 ± 0.0163,0.9391 ± 0.0083,0.9861 ± 0.0041
1,Regressão Logística,0.9228 ± 0.0121,0.9204 ± 0.0231,0.8807 ± 0.0185,0.9000 ± 0.0153,0.9705 ± 0.0069
2,Árvore de Decisão,0.9136 ± 0.0109,0.8865 ± 0.0209,0.8959 ± 0.0191,0.8910 ± 0.0134,0.9105 ± 0.0110
3,k-NN padronizado,0.9068 ± 0.0040,0.9006 ± 0.0164,0.8586 ± 0.0101,0.8790 ± 0.0038,0.9530 ± 0.0069


### 32. Comparação com o baseline

O baseline da T5-A2 obteve aproximadamente **60,60% de acurácia**, mas `recall = 0` para spam.

A célula abaixo compara diretamente a acurácia média dos modelos da validação cruzada com esse baseline.


In [29]:
baseline_accuracy_reference = baseline_accuracy

comparison_to_baseline = cv_summary[
    ["modelo", "accuracy_media", "accuracy_dp"]
].copy()

comparison_to_baseline["ganho_abs_sobre_baseline"] = (
    comparison_to_baseline["accuracy_media"] - baseline_accuracy_reference
)

display(
    comparison_to_baseline.style.format({
        "accuracy_media": "{:.4f}",
        "accuracy_dp": "{:.4f}",
        "ganho_abs_sobre_baseline": "{:+.4f}",
    })
)


,modelo,accuracy_media,accuracy_dp,ganho_abs_sobre_baseline
0,Random Forest,0.9527,0.0061,+0.3468
1,Regressão Logística,0.9228,0.0121,+0.3169
2,Árvore de Decisão,0.9136,0.0109,+0.3076
3,k-NN padronizado,0.9068,0.0040,+0.3008


### 33. Interpretação dos resultados

Nesta etapa, **não escolheremos ainda o modelo final apenas pela maior acurácia**.

A análise deve considerar conjuntamente:

- desempenho médio;
- variação entre folds;
- precision e recall da classe spam;
- F1;
- ROC-AUC;
- custo computacional;
- e, principalmente, o requisito posterior do trabalho: **no máximo 1% dos e-mails legítimos podem ser marcados como spam**.

Esse último requisito torna a seleção do limiar operacional tão importante quanto a comparação entre os classificadores.

Também é importante lembrar que o conjunto `X_test` / `y_test` permanece sem uso nesta etapa.


### Checkpoint T5-A4

Ao executar esta etapa no Google Colab, registre:

- [ ] resultados dos 5 folds para cada modelo;
- [ ] média e desvio-padrão da acurácia;
- [ ] média e desvio-padrão de precision, recall, F1 e ROC-AUC;
- [ ] comparação com o baseline;
- [ ] confirmação de que o conjunto de teste final continua intocado.

**Próxima etapa — T5-A5:** comparar explicitamente o **k-NN com e sem padronização dos atributos**, usando as mesmas 5 dobras, e explicar tecnicamente a diferença.


---
## T5-A5 — k-NN com e sem padronização

O item 3 do trabalho solicita comparar o **k-NN com e sem padronização dos atributos** e explicar a diferença observada.

O k-NN classifica uma observação com base na proximidade em relação aos exemplos de treinamento. Portanto, a **escala numérica das features influencia diretamente o cálculo de distância**.

Nesta base há atributos com amplitudes muito diferentes. Por exemplo, algumas variáveis de frequência possuem valores relativamente pequenos, enquanto atributos relacionados a sequências de letras maiúsculas podem alcançar valores muito maiores.

Nesta etapa compararemos duas versões com o mesmo `k=5`:

1. **k-NN sem padronização**
2. **k-NN com `StandardScaler`**

Ambas serão avaliadas usando exatamente as **mesmas 5 dobras estratificadas** definidas anteriormente.


### 34. Definição das duas versões do k-NN

A única diferença entre os dois experimentos será a padronização.

No caso padronizado, o `StandardScaler` ficará dentro de um `Pipeline`, garantindo que média e desvio-padrão sejam estimados somente com os dados de treino de cada fold.


In [30]:
knn_variants = {
    "k-NN sem padronização": KNeighborsClassifier(
        n_neighbors=5,
    ),

    "k-NN com padronização": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(
            n_neighbors=5,
        )),
    ]),
}

for name, model in knn_variants.items():
    print(f"{name}:")
    print(model)
    print("-" * 80)


k-NN sem padronização:
KNeighborsClassifier()
--------------------------------------------------------------------------------
k-NN com padronização:
Pipeline(steps=[('scaler', StandardScaler()),
                ('model', KNeighborsClassifier())])
--------------------------------------------------------------------------------


### 35. Validação cruzada das duas versões

Usaremos novamente as métricas:

- accuracy
- precision
- recall
- F1
- ROC-AUC

e as mesmas 5 dobras de `cv`.


In [31]:
knn_fold_records = []

for model_name, model in knn_variants.items():
    print(f"Avaliando: {model_name}...")

    scores = cross_validate(
        estimator=model,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
    )

    for fold in range(N_SPLITS):
        knn_fold_records.append({
            "modelo": model_name,
            "fold": fold + 1,
            "accuracy": scores["test_accuracy"][fold],
            "precision": scores["test_precision"][fold],
            "recall": scores["test_recall"][fold],
            "f1": scores["test_f1"][fold],
            "roc_auc": scores["test_roc_auc"][fold],
            "fit_time_s": scores["fit_time"][fold],
        })

print("✓ Comparação do k-NN concluída.")


Avaliando: k-NN sem padronização...
Avaliando: k-NN com padronização...
✓ Comparação do k-NN concluída.


### 36. Resultados por fold


In [32]:
knn_folds_df = pd.DataFrame(knn_fold_records)

display(
    knn_folds_df.style.format({
        "accuracy": "{:.4f}",
        "precision": "{:.4f}",
        "recall": "{:.4f}",
        "f1": "{:.4f}",
        "roc_auc": "{:.4f}",
        "fit_time_s": "{:.3f}",
    })
)


,modelo,fold,accuracy,precision,recall,f1,roc_auc,fit_time_s
0,k-NN sem padronização,1,0.8084,0.7670,0.7379,0.7522,0.8733,0.023
1,k-NN sem padronização,2,0.8030,0.7526,0.7448,0.7487,0.8681,0.022
2,k-NN sem padronização,3,0.7812,0.7157,0.7379,0.7267,0.8456,0.027
3,k-NN sem padronização,4,0.7622,0.6898,0.7207,0.7049,0.8514,0.027
4,k-NN sem padronização,5,0.8030,0.7799,0.6966,0.7359,0.8644,0.009
5,k-NN com padronização,1,0.9117,0.9121,0.8586,0.8845,0.9559,0.042
6,k-NN com padronização,2,0.9062,0.8961,0.8621,0.8787,0.9611,0.036
7,k-NN com padronização,3,0.9062,0.9018,0.8552,0.8779,0.9486,0.042
8,k-NN com padronização,4,0.9090,0.9176,0.8448,0.8797,0.9559,0.037
9,k-NN com padronização,5,0.9008,0.8754,0.8724,0.8739,0.9437,0.049


### 37. Média e desvio-padrão

A tabela abaixo resume o desempenho das duas versões no formato exigido pelo trabalho.


In [33]:
knn_summary_rows = []

for model_name in knn_variants.keys():
    model_folds = knn_folds_df[knn_folds_df["modelo"] == model_name]

    row = {"modelo": model_name}

    for metric in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
        row[f"{metric}_media"] = model_folds[metric].mean()
        row[f"{metric}_dp"] = model_folds[metric].std(ddof=1)

    row["fit_time_media_s"] = model_folds["fit_time_s"].mean()
    knn_summary_rows.append(row)

knn_summary = pd.DataFrame(knn_summary_rows)

display(
    knn_summary.style.format({
        col: "{:.4f}"
        for col in knn_summary.columns
        if col != "modelo"
    })
)


,modelo,accuracy_media,accuracy_dp,precision_media,precision_dp,recall_media,recall_dp,f1_media,f1_dp,roc_auc_media,roc_auc_dp,fit_time_media_s
0,k-NN sem padronização,0.7916,0.0194,0.7410,0.0374,0.7276,0.0195,0.7337,0.0190,0.8606,0.0117,0.0215
1,k-NN com padronização,0.9068,0.0040,0.9006,0.0164,0.8586,0.0101,0.8790,0.0038,0.9530,0.0069,0.0416


### 38. Comparação compacta e ganho da padronização

Além de média e desvio-padrão, calcularemos o ganho absoluto da versão padronizada sobre a versão sem padronização.


In [34]:
knn_compact_rows = []

for _, row in knn_summary.iterrows():
    knn_compact_rows.append({
        "modelo": row["modelo"],
        "accuracy": f'{row["accuracy_media"]:.4f} ± {row["accuracy_dp"]:.4f}',
        "precision": f'{row["precision_media"]:.4f} ± {row["precision_dp"]:.4f}',
        "recall": f'{row["recall_media"]:.4f} ± {row["recall_dp"]:.4f}',
        "f1": f'{row["f1_media"]:.4f} ± {row["f1_dp"]:.4f}',
        "roc_auc": f'{row["roc_auc_media"]:.4f} ± {row["roc_auc_dp"]:.4f}',
    })

knn_compact = pd.DataFrame(knn_compact_rows)
display(knn_compact)

raw_row = knn_summary[knn_summary["modelo"] == "k-NN sem padronização"].iloc[0]
scaled_row = knn_summary[knn_summary["modelo"] == "k-NN com padronização"].iloc[0]

knn_gain = pd.DataFrame({
    "métrica": ["accuracy", "precision", "recall", "f1", "roc_auc"],
    "sem_padronizacao": [
        raw_row["accuracy_media"],
        raw_row["precision_media"],
        raw_row["recall_media"],
        raw_row["f1_media"],
        raw_row["roc_auc_media"],
    ],
    "com_padronizacao": [
        scaled_row["accuracy_media"],
        scaled_row["precision_media"],
        scaled_row["recall_media"],
        scaled_row["f1_media"],
        scaled_row["roc_auc_media"],
    ],
})

knn_gain["ganho_absoluto"] = (
    knn_gain["com_padronizacao"] - knn_gain["sem_padronizacao"]
)

display(
    knn_gain.style.format({
        "sem_padronizacao": "{:.4f}",
        "com_padronizacao": "{:.4f}",
        "ganho_absoluto": "{:+.4f}",
    })
)


,modelo,accuracy,precision,recall,f1,roc_auc
0,k-NN sem padronização,0.7916 ± 0.0194,0.7410 ± 0.0374,0.7276 ± 0.0195,0.7337 ± 0.0190,0.8606 ± 0.0117
1,k-NN com padronização,0.9068 ± 0.0040,0.9006 ± 0.0164,0.8586 ± 0.0101,0.8790 ± 0.0038,0.9530 ± 0.0069


,métrica,sem_padronizacao,com_padronizacao,ganho_absoluto
0,accuracy,0.7916,0.9068,+0.1152
1,precision,0.7410,0.9006,+0.1596
2,recall,0.7276,0.8586,+0.1310
3,f1,0.7337,0.8790,+0.1453
4,roc_auc,0.8606,0.9530,+0.0925


### 39. Por que a padronização afeta o k-NN?

O k-NN normalmente usa uma medida de distância, como a distância Euclidiana:

$$
d(x,z)=\sqrt{\sum_{j=1}^{p}(x_j-z_j)^2}
$$

Se uma feature varia, por exemplo, de `0` a `15.000`, enquanto outra varia de `0` a `1`, a primeira pode dominar numericamente a distância mesmo sem ser semanticamente mais importante.

O `StandardScaler` transforma cada atributo de acordo com:

$$
z=\frac{x-\mu}{\sigma}
$$

fazendo com que cada variável passe a ser expressa em número de desvios-padrão em relação à sua média.

Assim, a comparação de vizinhança deixa de ser dominada apenas pelas features com maior amplitude numérica.


### 40. Interpretação do experimento

A interpretação deve ser feita a partir dos valores efetivamente obtidos no Colab.

Se a versão padronizada apresentar melhor desempenho, isso fornece evidência de que as diferenças de escala entre os atributos prejudicavam a noção de proximidade usada pelo k-NN.

Caso a diferença seja pequena ou alguma métrica piore, isso também deve ser relatado: a padronização é teoricamente relevante para métodos baseados em distância, mas seu efeito quantitativo depende da estrutura concreta do dataset.

O ponto metodológico essencial é que a transformação foi realizada **dentro do pipeline e dentro de cada fold**, evitando data leakage.


### Checkpoint T5-A5

Ao executar esta etapa no Google Colab, registre:

- [ ] média e desvio-padrão do k-NN sem padronização;
- [ ] média e desvio-padrão do k-NN com padronização;
- [ ] ganho absoluto em accuracy, precision, recall, F1 e ROC-AUC;
- [ ] explicação da diferença observada em função da escala dos atributos.

**Próxima etapa — T5-A6:** ajustar o limiar de decisão para que **no máximo 1% dos e-mails legítimos sejam classificados como spam** e medir quanto spam ainda passa pelo filtro.
